# UIT-ViOCD + PhoBERT: khung Google Colab cho Sequence Classification


## 1. Cài thư viện


In [ ]:
!pip -q install -U transformers datasets accelerate evaluate pyvi scikit-learn sentencepiece



Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: --break-sysmtem-packages


## 2. Import thư viện và cấu hình


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
from pyvi import ViTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

os.environ["WANDB_DISABLED"] = "true"

SEED = 42
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = 128
DEMO_MODE = False          # True nếu muốn chạy nhanh thử
DEMO_TRAIN_SAMPLES = 1200  # chỉ dùng khi DEMO_MODE=True
DEMO_VAL_SAMPLES = 300
DEMO_TEST_SAMPLES = 300
OUTPUT_DIR = "./uit_viocd_phobert"

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 3. Tải dataset UIT-ViOCD từ Hugging Face


In [ ]:
raw_ds = load_dataset("tarudesu/ViOCD")
raw_ds


## 4. Xem nhanh cấu trúc dữ liệu


In [ ]:
for split in raw_ds.keys():
    print("=" * 80)
    print("SPLIT:", split)
    print("Columns:", raw_ds[split].column_names)
    print("Features:", raw_ds[split].features)
    print(raw_ds[split][0])


## 5. Tự chọn cột text và cột nhãn


In [ ]:
def pick_split_name(ds_dict, candidates):
    for name in candidates:
        if name in ds_dict:
            return name
    return None

train_split = pick_split_name(raw_ds, ["train", "training"])
val_split   = pick_split_name(raw_ds, ["validation", "valid", "dev"])
test_split  = pick_split_name(raw_ds, ["test"])

assert train_split is not None, "Không tìm thấy train split"

def choose_text_column(columns):
    segmented_candidates = [
        "sentence_segmented", "segmented_sentence", "review_segmented",
        "text_segmented", "comment_segmented", "sentence_preprocessed"
    ]
    raw_candidates = [
        "sentence", "review", "text", "comment", "content", "raw_text"
    ]
    for c in segmented_candidates:
        if c in columns:
            return c, True
    for c in raw_candidates:
        if c in columns:
            return c, False
    raise ValueError(f"Không tìm thấy cột text trong: {columns}")

def choose_label_column(columns):
    candidates = ["label", "labels", "target", "class", "y"]
    for c in candidates:
        if c in columns:
            return c
    raise ValueError(f"Không tìm thấy cột label trong: {columns}")

sample_columns = raw_ds[train_split].column_names
text_col, using_segmented_col = choose_text_column(sample_columns)
label_col = choose_label_column(sample_columns)

print("train_split =", train_split)
print("val_split   =", val_split)
print("test_split  =", test_split)
print("text_col    =", text_col)
print("label_col   =", label_col)
print("using_segmented_col =", using_segmented_col)


## 6. Chuẩn hóa dataset


In [ ]:
dataset_splits = {"train": raw_ds[train_split]}
if val_split is not None:
    dataset_splits["validation"] = raw_ds[val_split]
if test_split is not None:
    dataset_splits["test"] = raw_ds[test_split]

ds = DatasetDict(dataset_splits)

# Nếu muốn demo nhanh trên Colab
if DEMO_MODE:
    ds["train"] = ds["train"].shuffle(seed=SEED).select(range(min(DEMO_TRAIN_SAMPLES, len(ds["train"]))))
    if "validation" in ds:
        ds["validation"] = ds["validation"].shuffle(seed=SEED).select(range(min(DEMO_VAL_SAMPLES, len(ds["validation"]))))
    if "test" in ds:
        ds["test"] = ds["test"].shuffle(seed=SEED).select(range(min(DEMO_TEST_SAMPLES, len(ds["test"]))))

ds


## 7. Xử lý nhãn


In [ ]:
# Lấy danh sách nhãn
train_labels = ds["train"][label_col]

# Nếu label đã là int như 0/1
unique_labels = sorted(list(set(train_labels)))
print("Unique labels:", unique_labels)

# Gợi ý tên nhãn cho UIT-ViOCD
# Theo dữ liệu preview: 0 thường là non-complaint, 1 thường là complaint
if set(unique_labels) == {0, 1}:
    id2label = {0: "non_complaint", 1: "complaint"}
    label2id = {v: k for k, v in id2label.items()}
else:
    # fallback tổng quát
    id2label = {int(v): str(v) for v in unique_labels}
    label2id = {str(v): int(v) for v in unique_labels}

num_labels = len(id2label)

print("id2label:", id2label)
print("num_labels:", num_labels)


## 8. Chuẩn bị text cho PhoBERT


In [ ]:
def build_model_text(example):
    # Nếu dataset đã có cột segmented thì dùng luôn
    if using_segmented_col:
        text = str(example[text_col])
    else:
        # Nếu không có cột segmented thì tách từ bằng ViTokenizer
        text = ViTokenizer.tokenize(str(example[text_col]))
    return {
        "model_text": text,
        "labels": int(example[label_col]),
    }

processed_ds = ds.map(build_model_text)
processed_ds


## 9. Tokenizer


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_batch(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_ds = processed_ds.map(tokenize_batch, batched=True)

keep_cols = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_ds["train"].column_names:
    keep_cols.append("token_type_ids")

tokenized_ds = tokenized_ds.remove_columns(
    [c for c in tokenized_ds["train"].column_names if c not in keep_cols]
)

tokenized_ds.set_format("torch")
tokenized_ds


## 10. Kiểm tra 1 batch mẫu


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample_batch = [tokenized_ds["train"][i] for i in range(min(4, len(tokenized_ds["train"])))]
batch = data_collator(sample_batch)

for k, v in batch.items():
    print(k, v.shape)


## 11. Hàm đánh giá


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }


## 12. Khởi tạo model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)
model


## 13. TrainingArguments


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch" if "validation" in tokenized_ds else "no",
    save_strategy="epoch" if "validation" in tokenized_ds else "no",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True if "validation" in tokenized_ds else False,
    metric_for_best_model="f1_macro" if "validation" in tokenized_ds else None,
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
)


## 14. Trainer


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"] if "validation" in tokenized_ds else None,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 15. Train


In [ ]:
train_result = trainer.train()
train_result


## 16. Đánh giá trên validation / test


In [ ]:
if "validation" in tokenized_ds:
    val_metrics = trainer.evaluate(tokenized_ds["validation"])
    print("Validation metrics:")
    print(val_metrics)

if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(tokenized_ds["test"])
    print("Test metrics:")
    print(test_metrics)


## 17. In báo cáo chi tiết trên test


In [ ]:
if "test" in tokenized_ds:
    preds_output = trainer.predict(tokenized_ds["test"])
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    y_true = preds_output.label_ids

    target_names = [id2label[i] for i in range(num_labels)]
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0))

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))


## 18. Hàm dự đoán câu mới


In [ ]:
import torch.nn.functional as F

def preprocess_for_phobert(text: str) -> str:
    # Với PhoBERT, nên đưa text đã word-segment
    return ViTokenizer.tokenize(str(text).strip())

def predict_text(text: str):
    model.eval()
    text_proc = preprocess_for_phobert(text)

    inputs = tokenizer(
        text_proc,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
        pred_id = int(np.argmax(probs))

    result = {
        "input_text": text,
        "segmented_text": text_proc,
        "pred_id": pred_id,
        "pred_label": id2label[pred_id],
        "probs": {id2label[i]: float(probs[i]) for i in range(num_labels)},
    }
    return result


## 19. Demo dự đoán


In [ ]:
examples = [
    "sản phẩm đẹp, giao hàng nhanh, mình rất hài lòng",
    "đặt màu đen mà shop giao màu trắng, quá thất vọng",
    "ứng dụng hay nhưng hơi lag và thỉnh thoảng bị văng ra",
]

for text in examples:
    print("=" * 100)
    out = predict_text(text)
    print("Input:", out["input_text"])
    print("Segmented:", out["segmented_text"])
    print("Prediction:", out["pred_label"])
    print("Probabilities:", out["probs"])


## 20. Lưu model và tokenizer


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Đã lưu model vào:", OUTPUT_DIR)


## 21. Gợi ý chỉnh nhanh

- Muốn chạy nhanh hơn: bật `DEMO_MODE = True`
- Muốn kết quả tốt hơn:
  - tăng `num_train_epochs`
  - thử `MAX_LENGTH = 256`
  - thử batch size lớn hơn nếu GPU đủ
  - cân nhắc stratified split nếu bạn tự chia lại dữ liệu
- Muốn làm demo đẹp hơn:
  - thêm biểu đồ loss / accuracy
  - thêm giao diện Gradio nhỏ
